In [1]:
import time
import numpy as np

from b_Closed_form import BS_greeks
from c_MC import MC_BS_greeks_gpu, MC_heston_greeks_gpu
from d_FDM import FDM_BS_greeks, FDM_heston_greeks
from e_3_CVAE_greek import CVAE_greeks
from e_0_generate import generate_BS_params, generate_heston_params

In [91]:
#closed_form
# eta = (S0, K, r, sigma, T)

greek_key = ['delta', 'vega', 'rho', 'theta']
bs_eta = (1.0, 1.0, 0.03, np.sqrt(0.05), 1.5)
type = 'put'
relative_step = 0.01

vanilla = BS_greeks(bs_eta, type=type, relative_step=relative_step)
barrier = BS_greeks(bs_eta, B=0.5, type=type, relative_step=relative_step)

for key in greek_key:
    print(f"Vanilla Average {key}: {np.mean(vanilla[key]):.6f}")
    print(f"Barrier Average {key}: {np.mean(barrier[key]):.6f}")
    print(f"================================")

Vanilla Average delta: -0.381613
Barrier Average delta: -0.324582
Vanilla Average vega: 0.466928
Barrier Average vega: 0.281394
Vanilla Average rho: -0.701333
Barrier Average rho: -0.621566
Vanilla Average theta: -0.020776
Barrier Average theta: -0.008698


# MC

In [94]:
# MC bs
time_result = []
greek_result = {n: [] for n in ['delta', 'vega', 'rho', 'theta']}
greek_key = ['delta', 'vega', 'rho', 'theta']
bs_eta = (1.0, 1.0, 0.03, np.sqrt(0.05), 1.5)

for i in range(50):
    start = time.time()
    bs_result = MC_BS_greeks_gpu(
        bs_eta,
        B=0.5, #or None
        opt_type='put',
        n_paths=100_000,
        dt=0.001,
        seed=1234 + i*100,
        antithetic=True,
        relative_step=0.01
    )
    time_result.append(time.time() - start)
    for key in greek_key:
        greek_result[key].append(bs_result[key])

print(f"Average time: {np.mean(time_result):.6f}s")
for key in greek_key:
    print(f"Average {key}: {np.mean(greek_result[key]):.6f}")
    print(f"Std {key}: {np.std(greek_result[key]):.6f}")

Average time: 1.535718s
Average delta: -0.324859
Std delta: 0.003933
Average vega: 0.281002
Std vega: 0.017262
Average rho: -0.622350
Std rho: 0.009580
Average theta: -0.009281
Std theta: 0.003244


In [50]:
bs_result['details']['delta']['h'],bs_result['details']['vega']['h'],bs_result['details']['rho']['h'],bs_result['details']['theta']['h']

(np.float64(0.01), np.float64(0.00223606797749979), np.float64(0.0003), 0.015)

In [21]:
# MC hes
time_result = []
greek_result = {n: [] for n in ['delta', 'v0', 'rho', 'theta']}
greek_key = ['delta', 'v0', 'rho', 'theta']
hes_eta = (1.0, 1.0, 0.03, 2.0, 0.05, 0.5, -0.7, 0.04, 1.5)

for i in range(50):
    start = time.time()
    heston_result = MC_heston_greeks_gpu(
        hes_eta,
        B=0.3,
        opt_type='put',
        n_paths=100_000,
        dt=0.001,
        seed=1234 + i*100,
        antithetic=True,
        relative_step=0.01
    )
    time_result.append(time.time() - start)
    for key in greek_key:
        greek_result[key].append(heston_result[key])

print(f"Average time: {np.mean(time_result):.6f}s")
for key in greek_key:
    print(f"Average {key}: {np.mean(greek_result[key]):.6f}")
    print(f"Std {key}: {np.std(greek_result[key]):.6f}")

Average time: 2.940879s
Average delta: -0.278711
Std delta: 0.002793
Average v0: 0.257086
Std v0: 0.009821
Average rho: -0.540761
Std rho: 0.003715
Average theta: -0.014420
Std theta: 0.003359


In [52]:
heston_result['details']['delta']['h'],heston_result['details']['v0']['h'],heston_result['details']['rho']['h'],heston_result['details']['theta']['h']

(np.float64(0.01), np.float64(0.0004), np.float64(0.0003), 0.015)

# FDM

In [ ]:
# FDM
bs_eta = [1.0, 1.0, 0.03, np.sqrt(0.05), 1.5]
bs_greeks = FDM_BS_greeks(
    bs_eta,
    B=0.8,
    opt_type="put",
    solver_kwargs={"S_max": 4.0, "dS": 0.005, "dt": 0.0005},
)

{'S0': 0.01, 'r': 0.0003, 'sigma': 0.00223606797749979, 'T': 0.015}


In [67]:
print(bs_greeks['h'])
print(bs_greeks['delta'])
print(bs_greeks['vega'])
print(bs_greeks['rho'])
print(bs_greeks['theta'])

{'S0': 0.01, 'r': 0.0003, 'sigma': 0.00223606797749979, 'T': 0.015}
0.01522695264769081
-0.08877840196882691
-0.03725503403568971
0.007241889617288997


In [2]:
hes_eta = [1.0, 1.0, 0.03, 2.0, 0.05, 0.5, -0.7, 0.04, 1.5]
hes_greeks1 = FDM_heston_greeks(
    hes_eta,
    B=None,
    opt_type="call",
    greeks=("delta", "rho", "v0", "theta"),
    solver_kwargs={"S_max": 4.0, "v_max": 1.5, "dS": 0.01, "dv": 0.0001, "dt": 0.001}, # 
)

print(hes_greeks1['h'])
print(hes_greeks1['delta'])
print(hes_greeks1['v0'])
print(hes_greeks1['rho'])
print(hes_greeks1['theta'])

Calculating base
Calculating base finish
Calculating rho
{'S0': 0.01, 'r': 0.0003, 'v0': 0.0004, 'T': 0.015}
0.7037596781283975
0.317949267473229
0.8633567863232384
-0.04969401979270187


# CVAE

In [3]:
from e_2_CVAE import CVAE
import torch
if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

In [ ]:
# bs CVAE training settings
model_type = 'bs'
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
warmup_chunks = None # None or num

num_chunks  = 1358
cvae_type = "normal_weight" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 0.45
weight_alpha2 = None
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때


save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}_fine.pt"
result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}_fine"

ckpt = torch.load(save_path, map_location=device, weights_only=False)
cvae = CVAE(
    dim_x=ckpt["dim_x"],
    dim_eta=ckpt["dim_eta"],
    dim_z=ckpt["dim_z"],
    hidden_dims=ckpt["hidden_dims"],
    use_bn=ckpt.get("use_bn", False),
).to(device)

state_dict = ckpt["model_state"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

cvae.load_state_dict(state_dict)
cvae.eval()

bs_cvae, bs_ckpt = cvae, ckpt

In [98]:
# bs cvae
time_result = []
greek_key = ['delta', 'vega', 'rho', 'theta']
opt_key = ['van_call', 'van_put', 'barr_call', 'barr_put']
greek_result = {n: {m: [] for m in opt_key} for n in greek_key}

bs_eta = (1.0, 1.0, 0.03, np.sqrt(0.05), 1.5)
for i in range(50):
    start = time.time()
    result_bs = CVAE_greeks(
        cvae,
        ckpt,
        bs_eta,
        model_type=model_type,
        B=0.5,
        n_samples=100_000,
        #h={"delta": 0.01, "vega": 0.0004, "rho": 0.0003, "theta": 0.015},
        relative_step=0.01,
        antithetic=True,
        seed=1234 + i * 100,
    )
    time_result.append(time.time() - start)
    for key in greek_key:
        for opt in opt_key:
            greek_result[key][opt].append(result_bs["greeks"][key][opt])

print(f"Average time: {np.mean(time_result):.6f}s")
for key in greek_key:
    for opt in opt_key:
        print(f"Average {key} for {opt}: {np.mean(greek_result[key][opt]):.6f}")
    print("================================================================")

Average time: 0.315050s
Average delta for van_call: 0.626851
Average delta for van_put: -0.373663
Average delta for barr_call: 0.626851
Average delta for barr_put: -0.325863
Average vega for van_call: 0.475627
Average vega for van_put: 0.475788
Average vega for barr_call: 0.475628
Average vega for barr_put: 0.291769
Average rho for van_call: 0.742039
Average rho for van_put: -0.713829
Average rho for barr_call: 0.742038
Average rho for barr_put: -0.639350
Average theta for van_call: -0.050495
Average theta for van_put: -0.021389
Average theta for barr_call: -0.050495
Average theta for barr_put: -0.009672


In [26]:
# hes CVAE training settings
model_type = 'hes'
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 18
lr2         = 2e-6
l2          = 19
lr3         = 1e-6
l3          = 13
beta        = 1
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
warmup_chunks = None # None or num

num_chunks  = 2716 # 2425, 2716
cvae_type = "normal_weight" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 0.4
weight_alpha2 = None
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때

save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}"


ckpt = torch.load(save_path, map_location=device, weights_only=False)
cvae = CVAE(
    dim_x=ckpt["dim_x"],
    dim_eta=ckpt["dim_eta"],
    dim_z=ckpt["dim_z"],
    hidden_dims=ckpt["hidden_dims"],
    use_bn=ckpt.get("use_bn", False),
).to(device)

state_dict = ckpt["model_state"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

cvae.load_state_dict(state_dict)
cvae.eval()

CVAE(
  (recognition): RecognitionNet(
    (net): Sequential(
      (0): Linear(in_features=9, out_features=1024, bias=True)
      (1): Tanh()
      (2): Linear(in_features=1024, out_features=1024, bias=True)
      (3): Tanh()
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): Tanh()
      (6): Linear(in_features=512, out_features=256, bias=True)
      (7): Tanh()
    )
    (mu): Linear(in_features=256, out_features=2, bias=True)
    (logvar): Linear(in_features=256, out_features=2, bias=True)
  )
  (prior): PriorNet(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=1024, bias=True)
      (1): Tanh()
      (2): Linear(in_features=1024, out_features=1024, bias=True)
      (3): Tanh()
      (4): Linear(in_features=1024, out_features=512, bias=True)
      (5): Tanh()
      (6): Linear(in_features=512, out_features=256, bias=True)
      (7): Tanh()
    )
    (mu): Linear(in_features=256, out_features=2, bias=True)
    (logvar): Linear(in_feature

In [24]:
# hes full 
# CVAE training settings
model_type = 'hes_clip'
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 1024, 1024, 512, 256] # [128, 128, 64], [512, 256, 128], [1024, 512, 256], [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 26
lr2         = 2e-6
l2          = 27
lr3         = 1e-6
l3          = 13
beta        = 1
validation_chunk_idxs = [22,64,76,116,155,194]
val_every_chunks = 97
memory_on_gpu = True
warmup_chunks = None # None or num

num_chunks  = 4850
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = "barrier_put" # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0
weight_alpha2 = None
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때

if cvae_type == "base":
#     save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
#     result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}"
#     save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
#     result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}"
#     save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{l3}-{lr3}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
#     result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{l3}-{lr3}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}"

# over 4 layer 
#     save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
#     result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}"
    save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
    result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk{num_chunks}"
    
elif cvae_type == "barr_weight":
    save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{weight_alpha}_{weight_h}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
    result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{weight_alpha}_{weight_h}_{validation_chunk_idxs}_chunk{num_chunks}"
    

elif cvae_type == "add_put_loss" or cvae_type == "normal_weight":
#     save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}.pt"
#     result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
# _{batch_size}_{bn_chunks}_{lr}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}"
    save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}_fine.pt"
    result_base_path = f"result/cvae/{model_type}/cvae_{model_type}_{cvae_type}_{dim_z}_{hidden_dims[0]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{weight_mode}_{weight_alpha}_{validation_chunk_idxs}_chunk{num_chunks}_fine"


ckpt = torch.load(save_path, map_location=device, weights_only=False)
cvae = CVAE(
    dim_x=ckpt["dim_x"],
    dim_eta=ckpt["dim_eta"],
    dim_z=ckpt["dim_z"],
    hidden_dims=ckpt["hidden_dims"],
    use_bn=ckpt.get("use_bn", False),
).to(device)

state_dict = ckpt["model_state"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}

cvae.load_state_dict(state_dict)
cvae.eval()

CVAE(
  (recognition): RecognitionNet(
    (net): Sequential(
      (0): Linear(in_features=9, out_features=1024, bias=True)
      (1): Tanh()
      (2): Linear(in_features=1024, out_features=1024, bias=True)
      (3): Tanh()
      (4): Linear(in_features=1024, out_features=1024, bias=True)
      (5): Tanh()
      (6): Linear(in_features=1024, out_features=1024, bias=True)
      (7): Tanh()
      (8): Linear(in_features=1024, out_features=512, bias=True)
      (9): Tanh()
      (10): Linear(in_features=512, out_features=256, bias=True)
      (11): Tanh()
    )
    (mu): Linear(in_features=256, out_features=2, bias=True)
    (logvar): Linear(in_features=256, out_features=2, bias=True)
  )
  (prior): PriorNet(
    (net): Sequential(
      (0): Linear(in_features=7, out_features=1024, bias=True)
      (1): Tanh()
      (2): Linear(in_features=1024, out_features=1024, bias=True)
      (3): Tanh()
      (4): Linear(in_features=1024, out_features=1024, bias=True)
      (5): Tanh()
      (6)

In [27]:
# hes cvae
print(f"Heston checkpoint: {save_path}")
time_result = []
greek_key = ['delta', 'v0', 'rho', 'theta']
opt_key = ['van_call', 'van_put', 'barr_call', 'barr_put']
greek_result = {n: {m: [] for m in opt_key} for n in greek_key}

hes_eta = [1.0, 1.0, 0.03, 2.0, 0.05, 0.5, -0.7, 0.04, 1.5]
for i in range(50):
    start = time.time()
    result_hes = CVAE_greeks(
        cvae,
        ckpt,
        hes_eta,
        model_type=model_type,
        B=0.3,
        n_samples=100_000,
        #h={"delta": 0.01, "v0": 0.0004, "rho": 0.0003, "theta": 0.015},
        relative_step=0.01,
        antithetic=True,
        seed=1234 + i * 100,
    )
    time_result.append(time.time() - start)
    for key in greek_key:
        for opt in opt_key:
            greek_result[key][opt].append(result_hes["greeks"][key][opt])

print(f"Average time: {np.mean(time_result):.6f}s")
for key in greek_key:
    for opt in opt_key:
        print(f"Average {key} for {opt}: {np.mean(greek_result[key][opt]):.6f}")
    print("================================================================")

Heston checkpoint: result/cvae/hes/cvae_hes_normal_weight_2_1024_16384_None_18-2e-05_19-2e-06_1_barrier_put_0.4_[15, 24, 78]_chunk2716.pt
Average time: 0.319724s
Average delta for van_call: 0.705509
Average delta for van_put: -0.294554
Average delta for barr_call: 0.705509
Average delta for barr_put: -0.278746
Average v0 for van_call: 0.319534
Average v0 for van_put: 0.329684
Average v0 for barr_call: 0.319534
Average v0 for barr_put: 0.266447
Average rho for van_call: 0.845309
Average rho for van_put: -0.574084
Average rho for barr_call: 0.845309
Average rho for barr_put: -0.546324
Average theta for van_call: -0.049479
Average theta for van_put: -0.021289
Average theta for barr_call: -0.049479
Average theta for barr_put: -0.014099


# MAE

In [27]:
MC_DT = 0.001
bs_para = generate_BS_params(n_sets=100, seed=1234)
bs_para[:, 2] = np.round(bs_para[:, 2] / MC_DT) * MC_DT

In [18]:
bs_para = np.array([[0.03, np.sqrt(0.05), 1.5]])

In [28]:
# BS delta MAE: closed-form is the reference.
import pandas as pd

OPTION_SPECS = {
    "van_call": ("call", None),
    "van_put": ("put", None),
    "barr_call": ("call", 0.8),
    "barr_put": ("put", 0.8),
}

ratio = 0.01
paths = (1_000, 10_000, 100_000)
bs_delta_rows = []
for row_idx, (r, sigma, T) in enumerate(bs_para):
    T = round(float(T) / 0.001) * 0.001
    eta = (1.0, 1.0, float(r), float(sigma), T)
    bs_greek_h = {
        "delta": max(abs(eta[0]) * ratio, 1e-6),
        "vega": max(abs(eta[3]) * ratio, 1e-6),
        "rho": max(abs(eta[2]) * ratio, 1e-6),
        "theta": max(abs(eta[4]) * ratio, 1e-6),
    }
    seed = 1234 + row_idx * 100

    closed_delta = {
        option: BS_greeks(eta, type=opt_type, B=B, h=bs_greek_h)["delta"]
        for option, (opt_type, B) in OPTION_SPECS.items()
    }
    mc_delta = {
        n_path: {
            option: MC_BS_greeks_gpu(
                eta,
                B=B,
                opt_type=opt_type,
                greeks=("delta",),
                n_paths=n_path,
                dt=0.001,
                h=bs_greek_h,
                seed=seed,
                antithetic=True,
            )["delta"]
            for option, (opt_type, B) in OPTION_SPECS.items()
        }
        for n_path in paths
    }
    cvae_delta = {
        n_path: CVAE_greeks(
            bs_cvae,
            bs_ckpt,
            eta,
            model_type="bs",
            B=0.8,
            greeks=("delta",),
            n_samples=n_path,
            h=bs_greek_h,
            antithetic=True,
            seed=seed,
        )["greeks"]["delta"]
        for n_path in paths
    }

    for option in OPTION_SPECS:
        bs_delta_rows.append(
            {
                "set_idx": row_idx,
                "option": option,
                "opt_type": OPTION_SPECS[option][0],
                "barrier": OPTION_SPECS[option][1],
                "S0": eta[0],
                "K": eta[1],
                "r": eta[2],
                "sigma": eta[3],
                "T": eta[4],
                "closed_form_delta": closed_delta[option],
                "mc_1k_delta": mc_delta[1_000][option],
                "mc_10k_delta": mc_delta[10_000][option],
                "mc_100k_delta": mc_delta[100_000][option],
                "cvae_1k_delta": cvae_delta[1_000][option],
                "cvae_10k_delta": cvae_delta[10_000][option],
                "cvae_100k_delta": cvae_delta[100_000][option],
                "mc_1k_n_paths": 1_000,
                "mc_10k_n_paths": 10_000,
                "mc_100k_n_paths": 100_000,
                "mc_dt": 0.001,
                "mc_seed": seed,
                "cvae_1k_n_samples": 1_000,
                "cvae_10k_n_samples": 10_000,
                "cvae_100k_n_samples": 100_000,
                "cvae_seed": seed,
                "delta_h": bs_greek_h["delta"],
                "rho_h": bs_greek_h["rho"],
                "vega_h": bs_greek_h["vega"],
                "theta_h": bs_greek_h["theta"],
            }
        )

    if (row_idx + 1) % 10 == 0:
        print(f"BS: {row_idx + 1}/{len(bs_para)} completed")


BS: 10/100 completed
BS: 20/100 completed
BS: 30/100 completed
BS: 40/100 completed
BS: 50/100 completed
BS: 60/100 completed
BS: 70/100 completed
BS: 80/100 completed
BS: 90/100 completed
BS: 100/100 completed


In [ ]:
from pathlib import Path

bs_delta_results = pd.DataFrame(bs_delta_rows)
for method, sample_size in (
    ("mc", "1k"),
    ("mc", "10k"),
    ("mc", "100k"),
    ("cvae", "1k"),
    ("cvae", "10k"),
    ("cvae", "100k"),
):
    bs_delta_results[f"{method}_{sample_size}_abs_error"] = (
        bs_delta_results[f"{method}_{sample_size}_delta"]
        - bs_delta_results["closed_form_delta"]
    ).abs()

bs_delta_mae = (
    bs_delta_results
    .groupby("option", sort=False)[
        [
            "mc_1k_abs_error",
            "mc_10k_abs_error",
            "mc_100k_abs_error",
            "cvae_1k_abs_error",
            "cvae_10k_abs_error",
            "cvae_100k_abs_error",
        ]
    ]
    .mean()
    .rename(
        columns={
            "mc_1k_abs_error": "MC 1k MAE vs closed-form",
            "mc_10k_abs_error": "MC 10k MAE vs closed-form",
            "mc_100k_abs_error": "MC 100k MAE vs closed-form",
            "cvae_1k_abs_error": "CVAE 1k MAE vs closed-form",
            "cvae_10k_abs_error": "CVAE 10k MAE vs closed-form",
            "cvae_100k_abs_error": "CVAE 100k MAE vs closed-form",
        }
    )
)

mae_output_dir = Path("result/greeks/mae")
mae_output_dir.mkdir(parents=True, exist_ok=True)

first_eta = bs_delta_results.iloc[0]
file_stem = (
    f"{first_eta['r']:.5f}_{first_eta['sigma']:.5f}_{first_eta['T']:.5f}_"
    f"h_value_{first_eta['delta_h']:.5f}_{first_eta['rho_h']:.5f}_"
    f"{first_eta['vega_h']:.5f}_{first_eta['theta_h']:.5f}_bs_delta_mae"
)
details_path = mae_output_dir / f"{file_stem}_details.csv"
summary_path = mae_output_dir / f"{file_stem}_summary.csv"

bs_delta_results.to_csv(details_path, index=False)
bs_delta_mae.to_csv(summary_path)

display(bs_delta_mae)
print(f"Saved: {details_path}")
print(f"Saved: {summary_path}")

,MC 1k MAE vs closed-form,MC 10k MAE vs closed-form,MC 100k MAE vs closed-form,CVAE 1k MAE vs closed-form,CVAE 10k MAE vs closed-form,CVAE 100k MAE vs closed-form
option,,,,,,
van_call,0.024166,0.014383,0.010491,0.048960,0.042319,0.041120
van_put,0.013351,0.010036,0.009431,0.014155,0.012286,0.012219
barr_call,0.156412,0.049959,0.016289,0.273667,0.098901,0.070534
barr_put,0.012774,0.003955,0.001166,0.017156,0.007719,0.005825


Saved: result/greeks/mae/0.01915_0.76735_2.93900_h_value_0.01000_0.00019_0.00767_0.02939_bs_delta_mae_details.csv
Saved: result/greeks/mae/0.01915_0.76735_2.93900_h_value_0.01000_0.00019_0.00767_0.02939_bs_delta_mae_summary.csv


In [30]:
MC_DT = 0.001
hes_para = generate_heston_params(n_sets=100, seed=1234)
hes_para[:, 6] = np.round(hes_para[:, 6] / MC_DT) * MC_DT

In [ ]:
# Heston delta MAE: MC with 100,000 paths is the reference.
import pandas as pd

if "hes_cvae" not in globals() or "hes_ckpt" not in globals():
    raise RuntimeError("Run the Heston CVAE setup cell before this cell.")

OPTION_SPECS = {
    "van_call": ("call", None),
    "van_put": ("put", None),
    "barr_call": ("call", 0.8),
    "barr_put": ("put", 0.8),
}

ratio = 0.01
paths = (1_000, 10_000, 100_000)
hes_delta_rows = []
for row_idx, (r, kappa, long_var, xi, corr, v0, T) in enumerate(hes_para):
    T = round(float(T) / 0.001) * 0.001
    eta = (
        1.0, 1.0, float(r), float(kappa), float(long_var), float(xi), float(corr), float(v0), T, 
        )
    hes_greek_h = {
        "delta": max(abs(eta[0]) * ratio, 1e-6),
        "rho": max(abs(eta[2]) * ratio, 1e-6),
        "v0": max(abs(eta[7]) * ratio, 1e-6),
        "theta": max(abs(eta[8]) * ratio, 1e-6),
    }
    seed = 1234 + row_idx * 100

    mc_delta = {
        n_path: {
            option: MC_heston_greeks_gpu(
                eta,
                B=B,
                opt_type=opt_type,
                greeks=("delta",),
                n_paths=n_path,
                dt=0.001,
                h=hes_greek_h,
                seed=seed,
                antithetic=True,
            )["delta"]
            for option, (opt_type, B) in OPTION_SPECS.items()
        }
        for n_path in paths
    }
    cvae_delta = {
        n_path: CVAE_greeks(
            hes_cvae,
            hes_ckpt,
            eta,
            model_type="hes",
            B=0.8,
            greeks=("delta",),
            n_samples=n_path,
            h=hes_greek_h,
            antithetic=True,
            seed=seed,
        )["greeks"]["delta"]
        for n_path in paths
    }

    for option in OPTION_SPECS:
        hes_delta_rows.append(
            {
                "set_idx": row_idx,
                "option": option,
                "opt_type": OPTION_SPECS[option][0],
                "barrier": OPTION_SPECS[option][1],
                "S0": eta[0],
                "K": eta[1],
                "r": eta[2],
                "kappa": eta[3],
                "long_var": eta[4],
                "xi": eta[5],
                "corr": eta[6],
                "v0": eta[7],
                "T": eta[8],
                "mc_1k_delta": mc_delta[1_000][option],
                "mc_10k_delta": mc_delta[10_000][option],
                "mc_100k_delta": mc_delta[100_000][option],
                "cvae_1k_delta": cvae_delta[1_000][option],
                "cvae_10k_delta": cvae_delta[10_000][option],
                "cvae_100k_delta": cvae_delta[100_000][option],
                "mc_1k_n_paths": 1_000,
                "mc_10k_n_paths": 10_000,
                "mc_reference_n_paths": 100_000,
                "mc_dt": 0.001,
                "mc_seed": seed,
                "cvae_1k_n_samples": 1_000,
                "cvae_10k_n_samples": 10_000,
                "cvae_100k_n_samples": 100_000,
                "cvae_seed": seed,
                "delta_h": hes_greek_h["delta"],
                "rho_h": hes_greek_h["rho"],
                "v0_h": hes_greek_h["v0"],
                "theta_h": hes_greek_h["theta"],
            }
        )

    if (row_idx + 1) % 10 == 0:
        print(f"Heston: {row_idx + 1}/{len(hes_para)} completed")

Heston: 10/100 completed
Heston: 20/100 completed
Heston: 30/100 completed
Heston: 40/100 completed
Heston: 50/100 completed
Heston: 60/100 completed
Heston: 70/100 completed
Heston: 80/100 completed
Heston: 90/100 completed
Heston: 100/100 completed


In [34]:
from pathlib import Path

hes_delta_results = pd.DataFrame(hes_delta_rows)
for method, sample_size in (
    ("mc", "1k"),
    ("mc", "10k"),
    ("cvae", "1k"),
    ("cvae", "10k"),
    ("cvae", "100k"),
):
    hes_delta_results[f"{method}_{sample_size}_abs_error"] = (
        hes_delta_results[f"{method}_{sample_size}_delta"]
        - hes_delta_results["mc_100k_delta"]
    ).abs()

hes_delta_mae = (
    hes_delta_results
    .groupby("option", sort=False)[
        [
            "mc_1k_abs_error",
            "mc_10k_abs_error",
            "cvae_1k_abs_error",
            "cvae_10k_abs_error",
            "cvae_100k_abs_error",
        ]
    ]
    .mean()
    .rename(
        columns={
            "mc_1k_abs_error": "MC 1k MAE vs MC (100k)",
            "mc_10k_abs_error": "MC 10k MAE vs MC (100k)",
            "cvae_1k_abs_error": "CVAE 1k MAE vs MC (100k)",
            "cvae_10k_abs_error": "CVAE 10k MAE vs MC (100k)",
            "cvae_100k_abs_error": "CVAE 100k MAE vs MC (100k)",
        }
    )
)

mae_output_dir = Path("result/greeks/mae")
mae_output_dir.mkdir(parents=True, exist_ok=True)

first_eta = hes_delta_results.iloc[0]
file_stem = (
    f"{first_eta['r']:.5f}_{first_eta['v0']:.5f}_{first_eta['T']:.5f}_"
    f"h_value_{first_eta['delta_h']:.5f}_{first_eta['rho_h']:.5f}_"
    f"{first_eta['v0_h']:.5f}_{first_eta['theta_h']:.5f}_hes_delta_mae"
)
details_path = mae_output_dir / f"{file_stem}_details.csv"
summary_path = mae_output_dir / f"{file_stem}_summary.csv"

hes_delta_results.to_csv(details_path, index=False)
hes_delta_mae.to_csv(summary_path)

display(hes_delta_mae)
print(f"Saved: {details_path}")
print(f"Saved: {summary_path}")

,MC 1k MAE vs MC (100k),MC 10k MAE vs MC (100k),CVAE 1k MAE vs MC (100k),CVAE 10k MAE vs MC (100k),CVAE 100k MAE vs MC (100k)
option,,,,,
van_call,0.009919,0.003204,0.011031,0.008502,0.008725
van_put,0.007546,0.002476,0.008983,0.006947,0.006963
barr_call,0.036280,0.012301,0.040171,0.019195,0.013044
barr_put,0.019659,0.005931,0.024302,0.009491,0.006679


Saved: result/greeks/mae/0.01915_0.10041_2.43300_h_value_0.01000_0.00019_0.00100_0.02433_hes_delta_mae_details.csv
Saved: result/greeks/mae/0.01915_0.10041_2.43300_h_value_0.01000_0.00019_0.00100_0.02433_hes_delta_mae_summary.csv
